# Session 05 - Sampling distributions and the CLT
Richard Brooks | 29 September 2026

Run Setup, then follow Codes 1, 2, 3, 4, 5, 6, 7 and 8 in teaching order.
Codes 1, 2, 7 and 8 can be omitted during class when time is short.
The red blocks in the companion PDF identify calculations to write during class; all answers are supplied.

Main example: **warehouse order sizes and packing capacity**. This is a known discrete population model,
not a fitted model or the exponential waiting-time problem in the tutorial/assignment.
Transparent figures are saved beside this notebook in `session05_figures`.

## Setup — imports, colours and figure export

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t
from IPython.display import display

FIGURES = Path("session05_figures")
FIGURES.mkdir(exist_ok=True)
BLUE, LIGHT_BLUE, ORANGE = "#145DA0", "#62B5E5", "#E57922"
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.transparent": True})

def save_figure(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=180, bbox_inches="tight", transparent=True)
    plt.show()

## Code 1 — One sample gives one mean; repeated samples give a distribution
First work through the nine ordered pairs on paper: each observation is independently 2, 4 or 6,
each with probability $1/3$. There are two observations **per sample**, not nine.
The nine pairs enumerate all possible samples of that size.

Predict: can a sample mean be 3 even though an individual observation cannot be 3? **Yes: (2,4) or (4,2).**

In [ ]:
population_values = np.array([2, 4, 6])
pair_means = (population_values[:, None] + population_values[None, :]) / 2
display(pd.DataFrame(pair_means, index=population_values, columns=population_values))
values, counts = np.unique(pair_means, return_counts=True)
probabilities = counts / 9
display(pd.DataFrame({"sample mean": values, "probability": probabilities}))
print("E[mean] =", np.sum(values * probabilities))
print("Var(mean) =", np.sum((values - 4)**2 * probabilities))
print("P(mean <= 3) =", probabilities[values <= 3].sum())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), sharey=True)
axes[0].bar(population_values, [1/3]*3, width=.5, color=BLUE)
axes[1].bar(values, probabilities, width=.5, color=ORANGE)
for ax in axes:
    ax.set_xticks(range(2, 7))
    ax.set_ylabel("Probability")
axes[0].set(title="One observation", xlabel="Value of X")
axes[1].set(title="Mean of two independent observations", xlabel="Value of sample mean")
fig.tight_layout()
save_figure(fig, "01_exact_sampling")

The centre is 4 in both panels. The mean has variance $4/3$, versus $8/3$ for one observation.
These are **exact distributions**, not a normal approximation and not a simulation.

## Code 2 - Optional arithmetic check: plan the plate sample
Plate thickness has known population SD 4.2 micrometres.
Require $SE(\bar X)\le0.5$: $n\ge(4.2/0.5)^2=70.56$, hence **71 plates**.
Check the two neighbouring integers; no simulation is needed.

In [ ]:
sigma_plate = 4.2
target_se = .5
n_required = int(np.ceil((sigma_plate/target_se)**2))
print("Smallest sample size:", n_required)
for n in (n_required-1, n_required):
    print(f"n = {n}: SE = {sigma_plate/np.sqrt(n):.4f} micrometres")
assert n_required == 71
assert sigma_plate/np.sqrt(70) > target_se >= sigma_plate/np.sqrt(71)

## Code 3 — A normal population: exact probabilities for an average
Book example: individual plate thickness $X\sim N(250,4.2^2)$, in micrometres.
For 36 independent plates, $\bar X\sim N(250,0.7^2)$ **exactly**.

On paper: calculate $SE=4.2/\sqrt{36}=0.7$, then standardise 248.6 and 251.4 to −2 and 2.
Compare the probability for **one plate** with the probability for the **mean of 36 plates**.

In [ ]:
mu_plate, sigma_plate, n_plate = 250.0, 4.2, 36
se_plate = sigma_plate / np.sqrt(n_plate)
individual = norm(loc=mu_plate, scale=sigma_plate)
mean_plate = norm(loc=mu_plate, scale=se_plate)  # Notice that the standard deviation is the SE, not the SD of the population.
print(f"SE = {se_plate:.2f} micrometres")
print(f"P(248.6 < X < 251.4)     = {individual.cdf(251.4)-individual.cdf(248.6):.4f}")
print(f"P(248.6 < mean < 251.4)  = {mean_plate.cdf(251.4)-mean_plate.cdf(248.6):.4f}")
print(f"P(mean > 251.4)         = {mean_plate.sf(251.4):.4f}")
print(f"95th percentile of means = {mean_plate.ppf(.95):.3f} micrometres")

xs = np.linspace(235, 265, 800)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(xs, individual.pdf(xs), color=BLUE, label="One plate: SD = 4.2")
ax.plot(xs, mean_plate.pdf(xs), color=ORANGE, label="Mean of 36 plates: SE = 0.7")
ax.set(xlabel="Thickness / mean thickness (micrometres)", ylabel="Density")
ax.legend()
fig.tight_layout()
save_figure(fig, "03_normal_mean")

Expected results: **0.2611**, **0.9545**, **0.0228**, **251.151 µm**.
The last number is a percentile of hypothetical sample means under a known model, not a confidence limit for an unknown mean.
SciPy `scale` is the SD of whichever random quantity we model: 4.2 for $X$, 0.7 for $\bar X$.

## Code 4 - Warehouse orders and repeated-group means
Orders contain 1, 2 or 6 items, with probabilities 0.6, 0.3 and 0.1.
Assume independent orders from this stable mix.

On paper: $\mu=1(0.6)+2(0.3)+6(0.1)=1.8$ and
$\sigma^2=1^2(0.6)+2^2(0.3)+6^2(0.1)-1.8^2=2.16$.

We will simulate many independent groups of orders at several group sizes. In each group, `mean(axis=1)` calculates one sample mean.
A sample mean can be non-integer even though every individual order has an integer size.

In [ ]:
order_sizes = np.array([1, 2, 6])
order_probs = np.array([0.6, 0.3, 0.1])
mu = np.sum(order_sizes * order_probs)
variance = np.sum((order_sizes - mu)**2 * order_probs)
sigma = np.sqrt(variance)
print(f"Population mean = {mu:.1f} items; variance = {variance:.2f}; SD = {sigma:.4f} items")
# Model parameters for the repeated-group simulation below.

### Code 4 continued - Do larger groups have more stable averages?
Predict what changes for groups of 2, 10 and 50 orders.
We simulate 12,000 independent groups of each size.

The first panel is a **PMF** (probability); the other panels are **density histograms** of means.
Means remain discrete at finite $n$. The normal curve is a smooth approximation, not their exact PMF.

In [ ]:
n_rep = 12_000
sample_sizes = (2, 10, 50)
rng = np.random.default_rng(51)
means, totals = {}, {}
rows = []
for n in sample_sizes:
    samples = rng.choice(order_sizes, size=(n_rep, n), p=order_probs)
    totals[n] = samples.sum(axis=1)
    means[n] = samples.mean(axis=1)
    rows.append({"Orders per group": n, "Mean of group means": means[n].mean(),
                 "Predicted SE": sigma/np.sqrt(n), "SD of group means": means[n].std(ddof=1)})
display(pd.DataFrame(rows).round(4))

fig, axes = plt.subplots(2, 2, figsize=(10, 6.2), sharex=True)
axes[0, 0].bar(order_sizes, order_probs, width=.35, color=BLUE)
axes[0, 0].set(title="One order: exact PMF", ylabel="Probability", xlabel="Items")
grid = np.linspace(.5, 6.5, 900)
for ax, n in zip(axes.flat[1:], sample_sizes):
    # Bin widths are multiples of the lattice spacing 1/n.
    step = max(1, n//10)
    bins = np.arange(n-.5, 6*n+step, step)/n
    ax.hist(means[n], bins=bins, density=True, color=LIGHT_BLUE, alpha=.8)
    ax.plot(grid, norm.pdf(grid, loc=mu, scale=sigma/np.sqrt(n)), color=ORANGE, label="Normal approximation")
    ax.axvline(mu, color=BLUE, ls=":", lw=1)
    ax.set(title=f"Mean of {n} orders; SE = {sigma/np.sqrt(n):.3f}",
           xlabel="Mean items per order", ylabel="Density")
    ax.legend(fontsize=8)
for ax in axes.flat:
    ax.set_xlim(.5, 6.5)
fig.tight_layout()
save_figure(fig, "04_clt")

Centre stays near 1.8; SE decreases; the shape becomes more nearly normal.
Individual orders still contain only 1, 2 or 6 items.

12,000 is the number of repeated groups, **not** the $n$ in $\sigma/\sqrt n$.
More repetitions stabilise the histogram; more orders per group reduce the SE.

## Code 5 - Packing capacity: now the random quantity is a total
There are 50 orders. Can a station with capacity for 110 items handle the group?

$T=X_1+\cdots+X_{50}$, so $E[T]=50(1.8)=90$ and $SD(T)=\sqrt{50(2.16)}=\sqrt{108}$.
The continuous normal approximation gives
$P(T\le110)\approx\Phi((110-90)/\sqrt{108})\approx0.9729$.

Compare with the fraction of simulated groups whose **total** is at most 110.
The two answers need not match exactly: finite-sample normal approximation and simulation error both matter.
We use the direct CLT approximation here; no continuity correction is introduced.

In [ ]:
n_orders, capacity = 50, 110
mean_total = n_orders * mu
sd_total = sigma * np.sqrt(n_orders)
approx_capacity = norm.cdf(capacity, loc=mean_total, scale=sd_total)
sim_capacity = np.mean(totals[n_orders] <= capacity) # Simulated proportion of groups with total <= capacity
print(f"Mean total = {mean_total:.1f}; SD of total = {sd_total:.4f} items")
print(f"Normal approximation: P(total <= 110) = {approx_capacity:.4f}")
print(f"Simulated proportion within capacity = {sim_capacity:.4f}")
print(f"Same event for the mean: mean <= {capacity/n_orders:.1f} items per order")
assert np.array_equal(totals[n_orders] <= capacity, means[n_orders] <= capacity/n_orders)

The total is discrete and right-skewed; its normal approximation is not exact.
More simulated groups reduce simulation noise, not the approximation error.
Both results concern this assumed order mix, not a guarantee about a real warehouse.

## Code 6 - One growing sequence: the law of large numbers
Follow 2,000 successive orders. Recalculate the average after each order.
This is **one growing sequence**, not thousands of independent fixed-size groups.

In [ ]:
# Create a random-number generator. The seed 52 makes the demonstration reproducible.
rng_lln = np.random.default_rng(52)

# Simulate 2,000 order sizes from the model defined in Code 4: 1, 2, or 6 items, with their specified probabilities.
sequence = rng_lln.choice(order_sizes, size=2_000, p=order_probs)

# Create the numbers 1 through 2,000: how many orders have been observed at each step.
sample_counts = np.arange(1, len(sequence)+1)

#Calculates the average after every order. np.cumsum gives the cumulative totals; dividing each total by the number observed gives the running averages.
running_mean = np.cumsum(sequence) / sample_counts

# Plot the running mean as a function of the number of orders observed so far. The dashed line shows the population mean.
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sample_counts, running_mean, color=BLUE, lw=1)
ax.axhline(mu, color=ORANGE, ls="--", label="Population mean = 1.8 items")
ax.set(xlabel="Orders observed so far", ylabel="Running mean (items per order)")
ax.legend()
fig.tight_layout()
save_figure(fig, "06_lln")

The running mean tends to settle near 1.8, but not monotonically.
Adjacent points share almost all their observations.
LLN: concentration near the mean. CLT: approximate shape of the sampling distribution.

## Code 7 — Short bridge to Session 6: estimate the SE and introduce t
Usually $\sigma$ is unknown. Use $s/\sqrt n$ to estimate the SE, with sample SD calculated using `ddof=1`.
This does not make the normal-reference calculation exact.
Under i.i.d. **normal** sampling, $(\bar X-\mu)/(S/\sqrt n)\sim t_{n-1}$ exactly.
For non-normal populations, any practical use needs a separate adequacy judgment.

In [ ]:
observed = np.array([8., 10., 12., 14.])
s = observed.std(ddof=1) # Sample standard deviation
print(f"Mean = {observed.mean():.1f}; sample variance = {s**2:.4f}")
print(f"Sample SD = {s:.4f}; estimated SE = {s/np.sqrt(len(observed)):.4f}")
print(f"Book's battery example: t = {(102-100)/(12/np.sqrt(16)):.4f}, df = 15")
grid_t = np.linspace(-4, 4, 600)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(grid_t, norm.pdf(grid_t), color=BLUE, label="Standard normal")
ax.plot(grid_t, t.pdf(grid_t, df=3), color=ORANGE, label="t, df = 3")
ax.plot(grid_t, t.pdf(grid_t, df=15), color=LIGHT_BLUE, label="t, df = 15")
ax.set(xlabel="Standardised value", ylabel="Density")
ax.legend()
fig.tight_layout()
save_figure(fig, "07_t_bridge")

Student's t has heavier tails because the denominator is estimated and varies between samples.
It approaches the standard normal as the degrees of freedom increase. Session 6 uses this for mean confidence intervals.
No confidence interval or hypothesis test is constructed in this demonstration.

## Code 8 - Optional bridge: a proportion is an average
Let $Y=1$ for a defective component and $0$ otherwise, with known $p=0.04$.
For 1,000 independent components, $SE(\hat p)=\sqrt{0.04(0.96)/1000}=0.00620$,
or **0.620 percentage points**.
The short illustrative vector below shows why a proportion is a mean; it is not a simulated study of 1,000 components.

In [ ]:
illustrative = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0])
print("Illustration: count/n =", illustrative.sum()/len(illustrative))
print("Illustration: mean =", illustrative.mean())
p_defect, n_components = .04, 1000
print(f"Model SE for 1000 components = {np.sqrt(p_defect*(1-p_defect)/n_components):.5f}")

### Instructor checks and scope
The exact pair, plate and battery examples come from Chapter 5. The order-size model is a separate lecture example.
Tutorial 5 and Assignment 3 are deliberately left unchanged: students transfer the methods to their own contexts.
No assignment dataset, exponential simulation, assignment solution sequence or gamma-tail check is included here.

In [ ]:
assert np.isclose(order_probs.sum(), 1)
assert np.isclose(mu, 1.8)
assert np.isclose(variance, 2.16)
assert np.isclose(se_plate, .7)
assert np.isclose(np.sum(values * probabilities), 4)
assert np.isclose(np.sum((values-4)**2 * probabilities), 4/3)
print("All numerical checks passed.")